In [1]:
import numpy as np
import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
from probeinterface import get_probe, write_probeinterface

from format_waveform_data import map_contacts_to_intan

In [ ]:
''' File Paths '''
root_dir = "C:/Users/Isabel/Documents/data_temp/"

# session params
bird_id = "LMN88"
session_id = "260714"
ephys_id = "LMN88_260714_144143"
cmr = True

# path to .rhd intan file
intan_folder = f"{root_dir}{bird_id}_{session_id}/{ephys_id}/"

# kilosort directory
ks_dir = f"{intan_folder}kilosort4_bc"

# probe map
map_file_path = "Z:/Isabel/ephys/channel_maps/SILICON PROBE MAP H5_spikesort.xlsx"

# to save files
sort_dir = f"{intan_folder}kilosort4_bc/"
os.makedirs(sort_dir, exist_ok=True)

In [ ]:
''' set the probe layout '''
recording = se.read_intan(f"{intan_folder}info.rhd", stream_id='0')
fs = recording.get_sampling_frequency()

probe = get_probe(manufacturer="cambridgeneurotech", probe_name="ASSY-236-H5")
contact_sort, ch_names, shank_idx = map_contacts_to_intan(probe, map_file_path)
probe = probe.get_slice(contact_sort)
probe.set_device_channel_indices(np.arange(contact_sort.shape[0]))

recording = recording.set_probe(probe)
recording.set_property("channel_name", ch_names)
recording.set_property("group", shank_idx)

In [ ]:
''' find and exclude noisy channels '''
# check for noisy channels
noise_level = si.get_noise_levels(spre.common_reference(spre.highpass_filter(recording)))
plt.plot(noise_level)

In [ ]:
# set noise threshold and remove noisy channels
noise_thresh = 16
keep_ch_idx = noise_level < noise_thresh
print(f"excluding {np.sum(~keep_ch_idx)} channel(s): "
      f"{recording.channel_ids[~keep_ch_idx]}")
recording = recording.select_channels(recording.channel_ids[keep_ch_idx])

In [ ]:
# reset probe device indices after dropping channels, same pattern as before
probe = recording.get_probe()
probe.set_device_channel_indices(np.arange(probe.get_contact_count()))
recording = recording.set_probe(probe)
assert recording.get_num_channels() == recording.get_probe().get_contact_count()

n_channels_final = recording.get_num_channels()

In [ ]:
# check the gain
gains = recording.get_channel_gains()
assert np.allclose(gains, gains[0]), "expected uniform gain across channels"
gain_to_uV = float(gains[0])   # typically 0.195 for Intan RHD-based systems

In [ ]:
# common median reference if needed
if cmr:
    rec_cmr = spre.common_reference(recording, reference="global", operator="median")
else:
    rec_cmr = recording

In [ ]:
# save as binary for KS4 and BC
# todo check this
raw_file_path = f"{sort_dir}recording.bin"
rec_cmr.save(format="binary", folder=sort_dir, dtype="int16",
             file_paths=[raw_file_path], n_jobs=8, chunk_duration="1s")

In [ ]:
# save the probe mapping
write_probeinterface(f"{sort_dir}probe.json", probe)

In [ ]:
print(f"Binary ready for native Kilosort4 GUI: {raw_file_path}")
print(f"Channels: {n_channels_final}, fs: {fs}, gain_to_uV: {gain_to_uV}")